# 🏪 Cerebal/Solari — Final Championship Notebook
### Rossmann Store Sales 1st Place Solution (Gert Jacobusse) — Fully Adapted

**PDF Source Confirmed Techniques:**
| Technique | Source Quote | Status |
|---|---|---|
| `colsample_bytree=0.3` | *"changed from 0.7 to 0.3 because I had a lot of features"* | ✅ Implemented |
| Hub ratio features | *"ratio of sales during promotions/holidays/Saturdays"* | ✅ Implemented |
| Harmonic mean ensemble | *"for final ensembling I applied the harmonic mean"* | ✅ Implemented |
| 0.985 multiplier | *"should have applied 0.985 to achieve score well below 0.10"* | ✅ Implemented |
| May-Sept specialist | *"separate models trained only on months May to September"* | ✅ Implemented |
| Hub trend (Ridge) | *"fit a store specific linear model on day_number, weekday, promotions"* | ✅ Implemented |
| YoY lags | *"last quarter, last half year, last year, last 2 years"* | ✅ Implemented |
| Promo day counters | *"day counter for promotion cycle (every 14 days)"* | ✅ Implemented |
| Two-stage Customers | *"calculated recent data on number of customers instead of sales"* | ✅ Implemented |

## 📦 0. Package Check & Imports

In [ ]:
import subprocess, sys
for pkg in ['lightgbm', 'xgboost', 'catboost', 'seaborn', 'scipy']:
    try:
        __import__(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  ✓ {pkg} installed')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, shutil
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

pd.set_option('display.max_columns', 50)
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
})
PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']
SEED = 42
np.random.seed(SEED)
DATA_PATH = './Dataset/'
print('\nAll imports done ✓')

## 📂 1. Load Data

In [ ]:
train  = pd.read_csv(DATA_PATH + 'orders_train.csv')
test   = pd.read_csv(DATA_PATH + 'orders_test.csv')
meta   = pd.read_csv(DATA_PATH + 'hub_metadata.csv')
sample = pd.read_csv(DATA_PATH + 'sample_submission.csv')

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print(f'Train : {train.shape[0]:>7,} rows  |  {train["Date"].min().date()} → {train["Date"].max().date()}')
print(f'Test  : {test.shape[0]:>7,} rows  |  {test["Date"].min().date()} → {test["Date"].max().date()}')
print(f'Meta  : {meta.shape[0]:>7,} rows  |  {meta.shape[1]} columns')
print(f'Unique hubs: {train["HubID"].nunique()}')
train.head(3)

## ⚙️ 2. Feature Engineering
### 2.1 Base Calendar + Metadata Features

In [ ]:
LOYALTY_MONTHS = {
    'a': [1,4,7,10], 'b': [2,5,8,11], 'c': [3,6,9,12],
    'd': [1,3,5,7,9,11], 'e': [2,4,6,8,10,12],
}

def engineer_base(df, meta_df):
    df = df.merge(meta_df, on='HubID', how='left')
    df['Year']        = df['Date'].dt.year
    df['Month']       = df['Date'].dt.month
    df['Week']        = df['Date'].dt.isocalendar().week.astype(int)
    df['DayOfYear']   = df['Date'].dt.dayofyear
    df['DayOfMonth']  = df['Date'].dt.day
    df['Quarter']     = df['Date'].dt.quarter
    df['IsSummer']    = df['Month'].isin([6,7,8]).astype(np.int8)   # test period months

    # Cyclical encoding
    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Week_sin']    = np.sin(2*np.pi*df['Week']/52)
    df['Week_cos']    = np.cos(2*np.pi*df['Week']/52)
    df['Weekday_sin'] = np.sin(2*np.pi*df['Weekday']/7)
    df['Weekday_cos'] = np.cos(2*np.pi*df['Weekday']/7)

    # Metadata
    df['HasCompetitor']      = df['CompetitorDistance'].notna().astype(int)
    df['CompetitorDist_log'] = np.log1p(df['CompetitorDistance'].fillna(0))
    df['HasLoyalty']         = df['LoyaltyProgramSinceYear'].notna().astype(int)
    df['LoyaltyInterval']    = df['LoyaltyProgramInterval'].map(
        {'a':0,'b':1,'c':2,'d':3,'e':4}).fillna(-1)

    df['CompetitorAge_months'] = (
        (df['Year']  - df['CompetitorOpenSinceYear'].fillna(df['Year'])) * 12 +
        (df['Month'] - df['CompetitorOpenSinceMonth'].fillna(df['Month']))
    ).clip(lower=0)
    df['LoyaltyAge_weeks'] = (
        (df['Year'] - df['LoyaltyProgramSinceYear'].fillna(df['Year'])) * 52 +
        (df['Week'] - df['LoyaltyProgramSinceWeek'].fillna(df['Week']))
    ).clip(lower=0)

    def loyalty_active(row):
        if row['HasLoyalty'] == 0: return 0
        iv = row['LoyaltyProgramInterval']
        return 1 if (iv in LOYALTY_MONTHS and row['Month'] in LOYALTY_MONTHS[iv]) else 0
    df['LoyaltyActiveMonth'] = df.apply(loyalty_active, axis=1).astype(np.int8)

    # Interactions
    df['AnyHoliday']        = (df['RegionalHoliday'] > 0).astype(int)
    df['Promo_x_Weekend']   = df['PromoActive'] * df['IsWeekend']
    df['Promo_x_Holiday']   = df['PromoActive'] * df['AnyHoliday']
    df['Promo_x_School']    = df['PromoActive'] * df['SchoolClosureFlag']
    df['Holiday_x_Weekend'] = df['AnyHoliday'] * df['IsWeekend']
    df['Loyalty_x_Promo']   = df['LoyaltyActiveMonth'] * df['PromoActive']
    return df

print('Engineering base features...')
train_fe = engineer_base(train, meta)
test_fe  = engineer_base(test,  meta)
print(f'Done. Train: {train_fe.shape}  Test: {test_fe.shape}')

### 2.2 🎯 PDF Feature: Hub-Level Ratio Aggregates
*"aggregates by store: ratio of sales during promotions/holidays/Saturdays, proportion school holidays, proportion of days open"*  
*— Gert Jacobusse, 1st Place Solution PDF*

In [ ]:
def add_hub_ratio_features(train_df, test_df):
    """
    Compute per-hub static ratio features from full training history.
    These capture each hub's unique business profile:
    - promo_sales_ratio:   how much MORE a hub sells during promotions vs normal days
    - holiday_sales_ratio: how much MORE a hub sells on holidays vs normal days
    - saturday_sales_ratio:how much MORE a hub sells on Saturdays vs other weekdays
    - school_hol_pct:      fraction of open days that are school closure days
    - open_rate:           fraction of total days hub is actually open
    """
    open_df = train_df[train_df['IsOpen'] == 1].copy()
    overall = open_df.groupby('HubID')['OrderVolume'].mean().rename('hub_mean_vol')

    # Promo lift ratio
    promo_mean = open_df[open_df['PromoActive']==1].groupby('HubID')['OrderVolume'].mean()
    hub_stats = pd.DataFrame({'hub_mean_vol': overall})
    hub_stats['promo_sales_ratio']   = (promo_mean / overall).fillna(1.0)

    # Holiday lift ratio
    hol_mean = open_df[open_df['AnyHoliday']==1].groupby('HubID')['OrderVolume'].mean()
    hub_stats['holiday_sales_ratio'] = (hol_mean / overall).fillna(1.0)

    # Saturday lift ratio (Weekday==6 in 1-indexed Mon=1 scheme)
    sat_mask = open_df['Weekday'].isin([6])
    sat_mean = open_df[sat_mask].groupby('HubID')['OrderVolume'].mean()
    hub_stats['saturday_sales_ratio'] = (sat_mean / overall).fillna(1.0)

    # School holiday proportion
    hub_stats['school_hol_pct'] = open_df.groupby('HubID')['SchoolClosureFlag'].mean()

    # Open rate (open days / total days in training)
    total_days = train_df.groupby('HubID')['Date'].count()
    open_days  = open_df.groupby('HubID')['Date'].count()
    hub_stats['open_rate'] = (open_days / total_days).fillna(0)

    # Clip extreme ratios
    for col in ['promo_sales_ratio','holiday_sales_ratio','saturday_sales_ratio']:
        hub_stats[col] = hub_stats[col].clip(0.5, 3.0)

    hub_stats = hub_stats.reset_index()
    train_df = train_df.merge(hub_stats, on='HubID', how='left')
    test_df  = test_df.merge(hub_stats, on='HubID', how='left')

    for col in hub_stats.columns:
        if col == 'HubID': continue
        train_df[col] = train_df[col].fillna(train_df[col].median())
        test_df[col]  = test_df[col].fillna(train_df[col].median())

    print(f'Hub ratio features added: {[c for c in hub_stats.columns if c != "HubID"]}')
    return train_df, test_df

print('Adding hub-level ratio features (from 1st place PDF)...')
train_fe, test_fe = add_hub_ratio_features(train_fe, test_fe)

### 2.3 🎯 PDF Feature: Hub Trend via Ridge Regression
*"fit a store specific linear model on 1) day number 2) day of week 3) promotions — use predicted slope as feature"*

In [ ]:
def add_hub_trend_features(train_df, test_df):
    """
    For each hub, fit a Ridge regression on the LAST YEAR of open-day sales,
    using [day_number, weekday, promo_active] as regressors.
    The fitted slope (coefficient on day_number) = hub's sales trend.
    The predicted value at test-set dates = trend-extrapolated baseline.
    This is the '3rd feature category' from the 1st place solution.
    """
    cutoff = train_df['Date'].max()
    lookback_start = cutoff - pd.Timedelta(days=365)

    recent = train_df[
        (train_df['IsOpen'] == 1) &
        (train_df['Date'] >= lookback_start)
    ].copy()
    recent['day_num'] = (recent['Date'] - lookback_start).dt.days

    hub_slopes    = {}
    hub_intercepts = {}

    for hub_id, grp in recent.groupby('HubID'):
        if len(grp) < 14:
            hub_slopes[hub_id]     = 0.0
            hub_intercepts[hub_id] = grp['OrderVolume'].mean()
            continue
        X_r = grp[['day_num','Weekday','PromoActive']].values.astype(float)
        y_r = np.log1p(grp['OrderVolume'].values)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_r)
        ridge = Ridge(alpha=1.0)
        ridge.fit(X_scaled, y_r)
        hub_slopes[hub_id]     = ridge.coef_[0]   # trend slope on day_number
        hub_intercepts[hub_id] = ridge.intercept_  # baseline level

    # Map to both train and test
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df['hub_trend_slope']     = train_df['HubID'].map(hub_slopes).fillna(0)
    train_df['hub_trend_intercept'] = train_df['HubID'].map(hub_intercepts).fillna(
        train_df['HubID'].map(recent.groupby('HubID')['OrderVolume'].mean()))
    test_df['hub_trend_slope']      = test_df['HubID'].map(hub_slopes).fillna(0)
    test_df['hub_trend_intercept']  = test_df['HubID'].map(hub_intercepts).fillna(
        test_df['HubID'].map(recent.groupby('HubID')['OrderVolume'].mean()))

    # Days into future from train cutoff (positive for test, negative/zero for train)
    train_df['days_from_cutoff'] = (train_df['Date'] - cutoff).dt.days
    test_df['days_from_cutoff']  = (test_df['Date']  - cutoff).dt.days

    # Trend-projected baseline for each row
    train_df['hub_trend_proj'] = np.expm1(
        (train_df['hub_trend_slope'] * train_df['days_from_cutoff'] +
         train_df['hub_trend_intercept']).clip(-5, 12))
    test_df['hub_trend_proj']  = np.expm1(
        (test_df['hub_trend_slope']  * test_df['days_from_cutoff']  +
         test_df['hub_trend_intercept']).clip(-5, 12))

    print(f'Trend slope range: [{min(hub_slopes.values()):.4f}, {max(hub_slopes.values()):.4f}]')
    print(f'Positive trend hubs: {sum(1 for v in hub_slopes.values() if v > 0)}')
    return train_df, test_df

print('Fitting per-hub Ridge trend models (1st place technique)...')
train_fe, test_fe = add_hub_trend_features(train_fe, test_fe)

### 2.4 🎯 PDF Feature: Promo Distance (Day Counters)

In [ ]:
def add_promo_distance(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns: test_tmp['OrderVolume'] = np.nan
    full = pd.concat([
        train_df[['HubID','Date','PromoActive']],
        test_tmp[['HubID','Date','PromoActive']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)

    results = []
    for hub_id, grp in full.groupby('HubID'):
        grp = grp.sort_values('Date').reset_index(drop=True)
        dates = grp['Date'].values; promo = grp['PromoActive'].values; n = len(grp)
        days_since = np.full(n, -1, dtype=np.float32)
        days_until = np.full(n, -1, dtype=np.float32)
        last_p = None
        for i in range(n):
            if promo[i] == 1: last_p = dates[i]
            if last_p is not None: days_since[i] = (dates[i]-last_p)/np.timedelta64(1,'D')
        next_p = None
        for i in range(n-1,-1,-1):
            if promo[i] == 1: next_p = dates[i]
            if next_p is not None: days_until[i] = (next_p-dates[i])/np.timedelta64(1,'D')
        grp['DaysSinceLastPromo'] = days_since
        grp['DaysUntilNextPromo'] = days_until
        results.append(grp[['HubID','Date','DaysSinceLastPromo','DaysUntilNextPromo']])

    promo_dist = pd.concat(results, ignore_index=True)
    cutoff = train_df['Date'].max()
    train_df = train_df.merge(promo_dist[promo_dist['Date']<=cutoff], on=['HubID','Date'], how='left')
    test_df  = test_df.merge(promo_dist[promo_dist['Date']>cutoff],  on=['HubID','Date'], how='left')
    return train_df, test_df

print('Promo distance features...')
train_fe, test_fe = add_promo_distance(train_fe, test_fe)
print(f'DaysSinceLastPromo mean: {train_fe["DaysSinceLastPromo"].mean():.1f}  ✓')

### 2.5 OpenAfterClosed — Pent-Up Demand

In [ ]:
def add_open_after_closed(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns: test_tmp['OrderVolume'] = np.nan
    full = pd.concat([
        train_df[['HubID','Date','IsOpen']],
        test_tmp[['HubID','Date','IsOpen']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)

    full['IsOpen_Lag1']     = full.groupby('HubID')['IsOpen'].shift(1)
    full['OpenAfterClosed'] = ((full['IsOpen']==1) & (full['IsOpen_Lag1']==0)).astype(np.int8)
    full['closed_streak']   = (full['IsOpen']==0).astype(int)
    full['ClosedDaysStreak'] = full.groupby('HubID')['closed_streak'].transform(
        lambda x: x.groupby((x!=x.shift()).cumsum()).transform('cumsum')
    ).shift(1).fillna(0)

    cols = ['HubID','Date','IsOpen_Lag1','OpenAfterClosed','ClosedDaysStreak']
    cutoff = train_df['Date'].max()
    train_df = train_df.merge(full[full['Date']<=cutoff][cols], on=['HubID','Date'], how='left')
    test_df  = test_df.merge(full[full['Date']>cutoff][cols],  on=['HubID','Date'], how='left')
    for col in ['IsOpen_Lag1','OpenAfterClosed','ClosedDaysStreak']:
        train_df[col] = train_df[col].fillna(0)
        test_df[col]  = test_df[col].fillna(0)
    return train_df, test_df

print('OpenAfterClosed features...')
train_fe, test_fe = add_open_after_closed(train_fe, test_fe)
reopened = train_fe[train_fe['OpenAfterClosed']==1]['OrderVolume'].mean()
normal   = train_fe[(train_fe['IsOpen']==1)&(train_fe['OpenAfterClosed']==0)]['OrderVolume'].mean()
print(f'Pent-up demand multiplier: {reopened/normal:.3f}x on reopen day  ✓')

### 2.6 🎯 PDF Feature: YoY Lags + Rolling (lag 364/365/371)

In [ ]:
def add_lag_rolling_features(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns: test_tmp['OrderVolume'] = np.nan
    if 'AppSessions' not in test_tmp.columns: test_tmp['AppSessions'] = np.nan

    full = pd.concat([
        train_df[['HubID','Date','OrderVolume','AppSessions','IsOpen']],
        test_tmp[['HubID','Date','OrderVolume','AppSessions','IsOpen']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)

    # Short/medium lags
    for lag in [7, 14, 21, 28, 42, 56]:
        full[f'vol_lag_{lag}'] = full.groupby('HubID')['OrderVolume'].shift(lag)

    # YoY lags — PDF: 'last quarter, last half year, last YEAR, last 2 years'
    for lag in [91, 182, 364, 365, 371, 728]:
        full[f'vol_lag_{lag}'] = full.groupby('HubID')['OrderVolume'].shift(lag)

    # YoY ratio: growth rate (PDF: 'year over year trend')
    full['vol_yoy_ratio_364'] = (full['vol_lag_7']+1) / (full['vol_lag_371']+1)
    full['vol_yoy_ratio_728'] = (full['vol_lag_7']+1) / (full['vol_lag_728']+1)

    # Rolling stats with shift-inside-transform (no leakage)
    for w in [7, 14, 28, 56, 91]:
        full[f'vol_roll{w}_mean'] = full.groupby('HubID')['OrderVolume'].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        full[f'vol_roll{w}_max']  = full.groupby('HubID')['OrderVolume'].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).max())

    full['vol_expand_mean'] = full.groupby('HubID')['OrderVolume'].transform(
        lambda x: x.shift(1).expanding().mean())
    full['vol_roll7_dev']   = full['vol_roll7_mean'] - full['vol_expand_mean']

    # AppSessions lags
    for lag in [7,14,28]:
        full[f'appsess_lag_{lag}'] = full.groupby('HubID')['AppSessions'].shift(lag)
    for w in [7,14,28]:
        full[f'appsess_roll{w}_mean'] = full.groupby('HubID')['AppSessions'].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean())

    lag_cols = [c for c in full.columns
                if c.startswith(('vol_lag','vol_roll','vol_expand','vol_yoy','appsess'))]
    cutoff = train_df['Date'].max()
    return (
        full[full['Date']<=cutoff][['HubID','Date']+lag_cols].copy(),
        full[full['Date']>cutoff][['HubID','Date']+lag_cols].copy(),
        lag_cols
    )

print('Computing YoY lags + rolling features...')
train_lags, test_lags, lag_cols = add_lag_rolling_features(train_fe, test_fe)
train_fe = train_fe.merge(train_lags, on=['HubID','Date'], how='left')
test_fe  = test_fe.merge(test_lags,  on=['HubID','Date'], how='left')
print(f'Lag columns: {len(lag_cols)}  (including lag_364, lag_365, lag_371, lag_728)')

### 2.7 Target Encodings (OOF Hub, Hub×Weekday, Hub×Month)

In [ ]:
def oof_hub_target_encode(train_df, test_df, target='OrderVolume', n_splits=5):
    train_sorted = train_df.sort_values('Date').copy()
    oof_vals = np.zeros(len(train_sorted))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for _, (tr_idx, val_idx) in enumerate(tscv.split(train_sorted)):
        hub_means   = train_sorted.iloc[tr_idx].groupby('HubID')[target].mean()
        global_mean = train_sorted.iloc[tr_idx][target].mean()
        oof_vals[val_idx] = train_sorted.iloc[val_idx]['HubID'].map(hub_means).fillna(global_mean).values
    train_df = train_df.copy()
    train_df['hub_oof_mean_vol'] = oof_vals
    hub_means_full = train_df.groupby('HubID')[target].mean()
    test_df = test_df.copy()
    test_df['hub_oof_mean_vol'] = test_df['HubID'].map(hub_means_full).fillna(train_df[target].mean())
    return train_df, test_df

def hub_weekday_encode(train_df, test_df, target='OrderVolume'):
    open_train = train_df[train_df['IsOpen']==1]
    hw = open_train.groupby(['HubID','Weekday'])[target].mean().reset_index()
    hw.columns = ['HubID','Weekday','hub_weekday_mean_vol']
    hub_mean = open_train.groupby('HubID')[target].mean()
    train_df = train_df.merge(hw, on=['HubID','Weekday'], how='left')
    test_df  = test_df.merge(hw, on=['HubID','Weekday'], how='left')
    train_df['hub_weekday_mean_vol'] = train_df['hub_weekday_mean_vol'].fillna(train_df['HubID'].map(hub_mean))
    test_df['hub_weekday_mean_vol']  = test_df['hub_weekday_mean_vol'].fillna(test_df['HubID'].map(hub_mean))
    return train_df, test_df

def hub_month_encode(train_df, test_df, target='OrderVolume'):
    open_train = train_df[train_df['IsOpen']==1].copy()
    hm = open_train.groupby(['HubID','Month'])[target].mean().reset_index()
    hm.columns = ['HubID','Month','hub_month_mean_vol']
    hub_mean = open_train.groupby('HubID')[target].mean()
    train_df = train_df.merge(hm, on=['HubID','Month'], how='left')
    test_df  = test_df.merge(hm, on=['HubID','Month'], how='left')
    train_df['hub_month_mean_vol'] = train_df['hub_month_mean_vol'].fillna(train_df['HubID'].map(hub_mean))
    test_df['hub_month_mean_vol']  = test_df['hub_month_mean_vol'].fillna(test_df['HubID'].map(hub_mean))
    return train_df, test_df

print('OOF hub encoding...')
train_fe, test_fe = oof_hub_target_encode(train_fe, test_fe)
print('Hub×Weekday encoding...')
train_fe, test_fe = hub_weekday_encode(train_fe, test_fe)
print('Hub×Month encoding...')
train_fe, test_fe = hub_month_encode(train_fe, test_fe)
print('All encodings done ✓')

### 2.8 Two-Stage: Predict AppSessions for Test

In [ ]:
def stage1_predict_appsessions(train_df, test_df):
    drop_s1 = ['Date','OrderVolume','AppSessions',
                'CompetitorOpenSinceMonth','CompetitorOpenSinceYear',
                'LoyaltyProgramSinceWeek','LoyaltyProgramSinceYear',
                'LoyaltyProgramInterval','CompetitorDistance','Id','closed_streak']
    train_s1 = train_df[(train_df['IsOpen']==1) & train_df['AppSessions'].notna()].copy()
    feat_cols = [c for c in train_s1.columns
                 if c not in drop_s1 and c != 'HubID'
                 and train_s1[c].dtype in [np.float64,np.float32,np.int64,np.int32,np.int8,bool]]
    key_lag = [f for f in ['appsess_lag_7','appsess_lag_14'] if f in feat_cols]
    train_s1_clean = train_s1.dropna(subset=key_lag).sort_values('Date').reset_index(drop=True)
    X_s1 = train_s1_clean[feat_cols].fillna(0)
    y_s1 = train_s1_clean['AppSessions'].values

    lgb_s1 = lgb.LGBMRegressor(
        objective='regression', n_estimators=1000, learning_rate=0.05,
        num_leaves=63, feature_fraction=0.6, bagging_fraction=0.8,
        bagging_freq=1, n_jobs=-1, verbosity=-1, random_state=SEED)
    lgb_s1.fit(X_s1, y_s1)

    test_open = test_df[test_df['IsOpen']==1].copy()
    X_test_s1 = test_open[feat_cols].fillna(0)
    test_df = test_df.copy()
    test_df['pred_AppSessions'] = 0.0
    test_df.loc[test_df['IsOpen']==1,'pred_AppSessions'] = np.maximum(lgb_s1.predict(X_test_s1), 0)
    train_df = train_df.copy()
    train_df['pred_AppSessions'] = train_df['AppSessions'].fillna(0)
    print(f'Stage 1 done. Test pred_AppSessions: mean={test_df["pred_AppSessions"].mean():.1f}')
    return train_df, test_df

print('Stage 1: predicting AppSessions for test...')
train_fe, test_fe = stage1_predict_appsessions(train_fe, test_fe)

### 2.9 Final Feature Matrix + NaN Diagnostics

In [ ]:
DROP_COLS = [
    'Date','OrderVolume','AppSessions',
    'CompetitorOpenSinceMonth','CompetitorOpenSinceYear',
    'LoyaltyProgramSinceWeek','LoyaltyProgramSinceYear',
    'LoyaltyProgramInterval','CompetitorDistance','Id','closed_streak',
]
train_model = train_fe[train_fe['IsOpen']==1].copy()
FEATURE_COLS = [c for c in train_model.columns
                if c not in DROP_COLS and c != 'HubID'
                and train_model[c].dtype in [np.float64,np.float32,np.int64,np.int32,np.int8,bool]]
if 'Month' not in FEATURE_COLS and 'Month' in train_model.columns:
    FEATURE_COLS.append('Month')

print(f'Total features: {len(FEATURE_COLS)}')
key_lags = [f'vol_lag_{d}' for d in [7,14,21] if f'vol_lag_{d}' in train_model.columns]
train_model_clean = train_model.dropna(subset=key_lags).sort_values('Date').reset_index(drop=True)
print(f'Train rows (open + lags available): {len(train_model_clean):,}')

X      = train_model_clean[FEATURE_COLS].fillna(0)
y      = np.log1p(train_model_clean['OrderVolume'])
y_raw  = train_model_clean['OrderVolume'].values
X_test = test_fe[FEATURE_COLS].fillna(0)

# Audit
nan_tr = X.isnull().sum(); nan_tr = nan_tr[nan_tr>0]
print(f'Train NaN: {"None" if nan_tr.empty else nan_tr.to_dict()}')
nan_te = X_test.isnull().sum(); nan_te = nan_te[nan_te>0]
print(f'Test NaN : {"None" if nan_te.empty else nan_te.to_dict()}')
print(f'\nX: {X.shape}  X_test: {X_test.shape}')
print(f'vol_lag_365 non-zero in test: {(X_test["vol_lag_365"]!=0).sum():,}/{len(X_test):,}  ✓')

## 📐 3. RMSLE + CV Setup

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# Covered mask: rows in at least one validation fold (fixes 3.569 ensemble bug)
covered_mask = np.zeros(len(X), dtype=bool)
for _, val_idx in tscv.split(X): covered_mask[val_idx] = True

print(f'RMSLE function ready.')
print(f'Naive mean baseline   : {rmsle(y_raw, np.full_like(y_raw, y_raw.mean(), dtype=float)):.5f}')
print(f'Covered rows          : {covered_mask.sum():,} / {len(X):,}')

## 🌲 4. LightGBM
*`feature_fraction=0.30` — PDF: "changed colsample_by_tree from 0.7 to 0.3 because I had a lot of features"*

In [ ]:
LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate'    : 0.02,
    'num_leaves'       : 127,
    'min_child_samples': 40,
    'feature_fraction' : 0.30,   # <-- KEY: from 1st place PDF
    'bagging_fraction' : 0.80,
    'bagging_freq'     : 1,
    'reg_alpha'        : 0.05,
    'reg_lambda'       : 1.0,
    'n_estimators'     : 5000,   # <-- KEY: from 1st place PDF (was 3000)
    'random_state'     : SEED,
    'verbosity'        : -1,
    'n_jobs'           : -1,
}

oof_lgb, pred_lgb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_lgb = []
lgb_fi = pd.Series(0.0, index=FEATURE_COLS)

print('Training LightGBM (feature_fraction=0.30)...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(500)])
    val_pred = np.maximum(np.expm1(model.predict(X.iloc[val_idx])), 0)
    oof_lgb[val_idx] = val_pred
    score = rmsle(y_raw[val_idx], val_pred)
    cv_scores_lgb.append(score)
    pred_lgb += np.maximum(np.expm1(model.predict(X_test)), 0) / N_SPLITS
    lgb_fi   += pd.Series(model.feature_importances_, index=FEATURE_COLS) / N_SPLITS
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration_}')

lgb_oof = np.mean(cv_scores_lgb)
print(f'\nLightGBM OOF RMSLE: {lgb_oof:.5f} ± {np.std(cv_scores_lgb):.5f}')

In [ ]:
# Feature importance — Rossmann features highlighted in red
top30 = lgb_fi.sort_values(ascending=False).head(30)
rossmann_feats = {'vol_lag_364','vol_lag_365','vol_lag_371','vol_lag_728',
                  'DaysSinceLastPromo','DaysUntilNextPromo','OpenAfterClosed',
                  'ClosedDaysStreak','promo_sales_ratio','holiday_sales_ratio',
                  'saturday_sales_ratio','school_hol_pct','open_rate',
                  'hub_trend_slope','hub_trend_proj','vol_yoy_ratio_364',
                  'LoyaltyActiveMonth','pred_AppSessions'}
fig, ax = plt.subplots(figsize=(11, 9))
bar_colors = [PALETTE[2] if f in rossmann_feats else PALETTE[0] for f in top30.index[::-1]]
ax.barh(top30.index[::-1], top30.values[::-1], color=bar_colors, alpha=0.9)
ax.set_title('Top 30 Feature Importances  (🔴 = from 1st place PDF)', fontsize=13)
ax.set_xlabel('Importance'); ax.grid(True, axis='x', alpha=0.4)
plt.tight_layout(); plt.show()

## 🚀 5. XGBoost (`colsample_bytree=0.30`)

In [ ]:
XGB_PARAMS = {
    'objective'        : 'reg:squarederror',
    'learning_rate'    : 0.02,
    'max_depth'        : 7,
    'n_estimators'     : 5000,
    'subsample'        : 0.80,
    'colsample_bytree' : 0.30,   # <-- KEY: from 1st place PDF
    'reg_alpha'        : 0.05,
    'reg_lambda'       : 1.0,
    'min_child_weight' : 40,
    'tree_method'      : 'hist',
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}

oof_xgb, pred_xgb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_xgb = []

print('Training XGBoost (colsample_bytree=0.30)...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=150, eval_metric='rmse')
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], verbose=500)
    val_pred = np.maximum(np.expm1(model.predict(X.iloc[val_idx])), 0)
    oof_xgb[val_idx] = val_pred
    score = rmsle(y_raw[val_idx], val_pred)
    cv_scores_xgb.append(score)
    pred_xgb += np.maximum(np.expm1(model.predict(X_test)), 0) / N_SPLITS
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration}')

xgb_oof = np.mean(cv_scores_xgb)
print(f'\nXGBoost OOF RMSLE: {xgb_oof:.5f} ± {np.std(cv_scores_xgb):.5f}')

## 🐱 6. CatBoost

In [ ]:
CAT_FEATURES = [c for c in ['HubFormat','AssortmentTier','LoyaltyProgram',
                              'IsOpen','PromoActive','RegionalHoliday','SchoolClosureFlag',
                              'Weekday','IsWeekend','AnyHoliday','LoyaltyActiveMonth',
                              'OpenAfterClosed','Quarter','IsSummer'] if c in FEATURE_COLS]
CB_PARAMS = {
    'iterations'           : 5000,
    'learning_rate'        : 0.02,
    'depth'                : 8,
    'l2_leaf_reg'          : 3,
    'rsm'                  : 0.30,   # colsample equivalent in CatBoost
    'random_strength'      : 1,
    'bagging_temperature'  : 1,
    'cat_features'         : CAT_FEATURES,
    'eval_metric'          : 'RMSE',
    'early_stopping_rounds': 150,
    'random_seed'          : SEED,
    'verbose'              : 500,
    'task_type'            : 'CPU',
    'task_type'            : 'CPU',
    'thread_count'         : -1,
}

oof_cb, pred_cb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_cb = []

print('Training CatBoost (rsm=0.30)...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_tr_cb, X_val_cb = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
    for col in CAT_FEATURES:
        X_tr_cb[col] = X_tr_cb[col].astype(str)
        X_val_cb[col] = X_val_cb[col].astype(str)
    model = cb.CatBoostRegressor(**CB_PARAMS)
    model.fit(X_tr_cb, y.iloc[tr_idx], eval_set=(X_val_cb, y.iloc[val_idx]))
    val_pred = np.maximum(np.expm1(model.predict(X_val_cb)), 0)
    oof_cb[val_idx] = val_pred
    score = rmsle(y_raw[val_idx], val_pred)
    cv_scores_cb.append(score)
    X_test_cb = X_test.copy()
    for col in CAT_FEATURES: X_test_cb[col] = X_test_cb[col].astype(str)
    pred_cb += np.maximum(np.expm1(model.predict(X_test_cb)), 0) / N_SPLITS
    print(f'  Fold {fold}: RMSLE={score:.5f}')

cb_oof = np.mean(cv_scores_cb)
print(f'\nCatBoost OOF RMSLE: {cb_oof:.5f} ± {np.std(cv_scores_cb):.5f}')

## ☀️ 7. May–September Specialist LightGBM
*PDF: "separate models trained only on months May to September — they cover the test set months"*  
*Test period is June–July 2015, so a summer-specialist model is well-calibrated.*

In [ ]:
# Filter train to May-September rows (months 5-9) across all years
summer_mask = train_model_clean['Month'].isin([5,6,7,8,9])
X_summer    = X[summer_mask].reset_index(drop=True)
y_summer    = y[summer_mask].reset_index(drop=True)
y_raw_summer = y_raw[summer_mask]
print(f'Summer specialist train rows: {len(X_summer):,} (May–Sep across all years)')

# Train 3-fold TimeSeriesSplit on summer data only
tscv_summer = TimeSeriesSplit(n_splits=3)
oof_summer  = np.zeros(len(X))
pred_summer = np.zeros(len(X_test))
cv_scores_summer = []

LGB_SUMMER = {**LGB_PARAMS, 'learning_rate': 0.03, 'n_estimators': 3000}

print('Training May-Sep specialist LightGBM...')
summer_idx = np.where(summer_mask)[0]

for fold, (tr_idx_s, val_idx_s) in enumerate(tscv_summer.split(X_summer), 1):
    model = lgb.LGBMRegressor(**LGB_SUMMER)
    model.fit(X_summer.iloc[tr_idx_s], y_summer.iloc[tr_idx_s],
              eval_set=[(X_summer.iloc[val_idx_s], y_summer.iloc[val_idx_s])],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)])
    val_pred = np.maximum(np.expm1(model.predict(X_summer.iloc[val_idx_s])), 0)
    # Map back to global OOF
    oof_summer[summer_idx[val_idx_s]] = val_pred
    score = rmsle(y_raw_summer[val_idx_s], val_pred)
    cv_scores_summer.append(score)
    pred_summer += np.maximum(np.expm1(model.predict(X_test)), 0) / 3
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration_}')


# CRITICAL: fill non-summer rows with oof_lgb so harmonic mean stays valid
# oof_summer was initialized to zeros; harmonic mean 1/sum(w/p) explodes when p=0
non_summer_mask = (oof_summer == 0)
oof_summer[non_summer_mask] = oof_lgb[non_summer_mask]

summer_oof = np.mean(cv_scores_summer)
print(f'\nSummer Specialist OOF RMSLE: {summer_oof:.5f} (on summer months only)')

## 🏆 8. Harmonic Mean Ensemble
*PDF: "for final ensembling I applied the harmonic mean"*  
*Harmonic mean stays conservative when models disagree — ideal for RMSLE which penalises over-prediction.*

In [ ]:
def weighted_harmonic_mean(preds_matrix, weights):
    """
    Compute weighted harmonic mean across models (columns).
    preds_matrix: shape (n_samples, n_models)  — must be positive
    weights:      shape (n_models,)
    Formula: sum(w) / sum(w_i / pred_i)
    """
    w = np.array(weights)
    w = w / w.sum()
    safe_preds = np.maximum(preds_matrix, 1e-9)
    return w.sum() / np.sum(w[np.newaxis,:] / safe_preds, axis=1)


# Stack OOF and test predictions (4 models: lgb, xgb, cb, summer)
oof_stack  = np.column_stack([oof_lgb, oof_xgb, oof_cb, oof_summer])
test_stack = np.column_stack([pred_lgb, pred_xgb, pred_cb, pred_summer])
model_names = ['LightGBM','XGBoost','CatBoost','Summer-Specialist']

# Optimize weights using harmonic mean on covered rows
def blend_rmsle_harmonic(weights):
    w = np.maximum(weights, 0)
    if w.sum() < 1e-9: return 999.0
    blend = weighted_harmonic_mean(oof_stack[covered_mask], w)
    return rmsle(y_raw[covered_mask], blend)

result = minimize(blend_rmsle_harmonic, x0=[0.4, 0.2, 0.2, 0.2],
                  method='Nelder-Mead', options={'maxiter': 3000, 'xatol': 1e-7})
weights = np.maximum(result.x, 0)
weights = weights / weights.sum()

oof_blend  = weighted_harmonic_mean(oof_stack,  weights)
test_blend = weighted_harmonic_mean(test_stack, weights)
blend_score = rmsle(y_raw[covered_mask], oof_blend[covered_mask])

print('Optimal ensemble weights (Harmonic Mean):')
for name, w in zip(model_names, weights):
    print(f'  {name}: {w:.4f}')

print(f'\nOOF RMSLE (covered rows):')
for name, scores in zip(model_names, [cv_scores_lgb,cv_scores_xgb,cv_scores_cb,cv_scores_summer]):
    print(f'  {name:<20}: {np.mean(scores):.5f}')
print(f'  {"Harmonic Ensemble":<20}: {blend_score:.5f}  ← should be lowest')

In [ ]:
# Compare arithmetic vs harmonic mean
arith_blend = oof_stack[covered_mask] @ weights
harm_blend  = oof_blend[covered_mask]
arith_score = rmsle(y_raw[covered_mask], arith_blend)
harm_score  = rmsle(y_raw[covered_mask], harm_blend)
print(f'Arithmetic mean ensemble RMSLE: {arith_score:.5f}')
print(f'Harmonic  mean ensemble RMSLE : {harm_score:.5f}')
print(f'Improvement from harmonic mean: {arith_score - harm_score:+.5f}')

# OOF diagnostic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
s_idx = np.random.choice(covered_mask.sum(), min(5000, covered_mask.sum()), replace=False)
y_c   = y_raw[covered_mask];  oof_c = oof_blend[covered_mask]
axes[0].scatter(y_c[s_idx], oof_c[s_idx], alpha=0.2, color=PALETTE[0], s=5)
mv = max(y_c.max(), oof_c.max())
axes[0].plot([0,mv],[0,mv],'r--',lw=1.5)
axes[0].set_title(f'Harmonic Ensemble OOF (RMSLE={harm_score:.5f})')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].grid(True, alpha=0.3)
residuals = np.log1p(np.maximum(oof_c,0)) - np.log1p(y_c)
axes[1].hist(residuals[s_idx], bins=80, color=PALETTE[3], alpha=0.85)
axes[1].axvline(0, color='red', ls='--')
axes[1].set_title('Residuals in Log Space')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 🔧 9. Post-Processing: 0.985 Multiplier
*PDF (exact quote): "I should have applied the mathematical estimate of 0.985 to achieve a private leaderboard score well below 0.10"*

In [ ]:
multipliers = np.arange(0.96, 1.01, 0.001)
mult_scores = [rmsle(y_c, oof_c * m) for m in multipliers]
best_mult   = multipliers[np.argmin(mult_scores)]
best_mult_score = np.min(mult_scores)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(multipliers, mult_scores, color=PALETTE[0], lw=2)
ax.axvline(best_mult, color=PALETTE[2], ls='--', lw=1.5, label=f'Best={best_mult:.3f}')
ax.axvline(0.985, color=PALETTE[1], ls=':', lw=1.5, label='PDF recommendation=0.985')
ax.set_title(f'RMSLE vs Multiplier  (Best={best_mult:.3f} → RMSLE={best_mult_score:.5f})')
ax.set_xlabel('Multiplier'); ax.set_ylabel('RMSLE')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Without multiplier : {blend_score:.5f}')
print(f'Best multiplier    : ×{best_mult:.3f}  → RMSLE={best_mult_score:.5f}')
print(f'PDF recommendation : ×0.985')
print(f'Gain               : {blend_score - best_mult_score:+.5f}')

## 📤 10. Generate Submission

In [ ]:
# Apply optimal multiplier
test_blend_final = test_blend * best_mult

# Zero out closed hubs (business rule: IsOpen=0 → always 0)
test_fe_reset = test_fe.reset_index(drop=True)
open_mask     = (test_fe_reset['IsOpen'] == 1).values
final_preds   = np.maximum(test_blend_final, 0)
final_preds[~open_mask] = 0.0

print(f'Total test rows  : {len(final_preds):,}')
print(f'Open (predicted) : {open_mask.sum():,}')
print(f'Closed (→0)      : {(~open_mask).sum():,}')
print(f'Multiplier       : ×{best_mult:.3f}')

# Copy sample → overwrite OrderVolume (preserves Id order & format exactly)
shutil.copy(DATA_PATH + 'sample_submission.csv', 'submission.csv')
submission = pd.read_csv('submission.csv')

# Align by Id (safe against any row ordering differences)
id_to_pred = dict(zip(test_fe_reset['Id'].values, final_preds))
submission['OrderVolume'] = submission['Id'].map(id_to_pred).fillna(0)
submission['OrderVolume'] = submission['OrderVolume'].clip(lower=0).round(2)

submission.to_csv('submission.csv', index=False)

print(f'\nSubmission stats:')
print(submission['OrderVolume'].describe())
print(f'Zeros: {(submission["OrderVolume"]==0).sum()} ({(submission["OrderVolume"]==0).mean()*100:.1f}%)')
print(f'\n✅ submission.csv saved!')
submission.head(10)

## 📊 11. Final Summary Dashboard

In [ ]:
rows = [
    ('LightGBM (feature_frac=0.30)', lgb_oof),
    ('XGBoost (colsample=0.30)',      xgb_oof),
    ('CatBoost (rsm=0.30)',           cb_oof),
    ('Summer Specialist',             summer_oof),
    ('Harmonic Ensemble',             blend_score),
    ('Harmonic Ensemble + ×Mult',     best_mult_score),
]
summary = pd.DataFrame(rows, columns=['Model','OOF RMSLE']).sort_values('OOF RMSLE')
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bar_cols = [PALETTE[1] if 'Mult' in m else (PALETTE[0] if 'Ensemble' in m else '#555')
            for m in summary['Model']]
bars = ax.bar(range(len(summary)), summary['OOF RMSLE'], color=bar_cols, alpha=0.87, width=0.6)
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary['Model'], rotation=20, ha='right', fontsize=9)
ax.set_title('OOF RMSLE by Model  (🟢 = submit this)', fontsize=12)
ax.set_ylabel('RMSLE (lower is better)')
ax.set_ylim(summary['OOF RMSLE'].min()*0.97, summary['OOF RMSLE'].max()*1.03)
ax.grid(True, axis='y', alpha=0.4)
for bar, val in zip(bars, summary['OOF RMSLE']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.0001, f'{val:.5f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

print(f"\n📌 Submit: submission.csv")
print(f"📌 Expected leaderboard range: 0.10 – 0.13 RMSLE")
print(f"📌 Rossmann 1st place individual models achieved: ~0.105 on private LB")